In [1]:
import requests
from dotenv import load_dotenv
import boto3
from pathlib import Path
import pandas as pd
import json

load_dotenv()  # reads .env, sets env variables

s3 = boto3.client("s3")
bucket_name = "weather-data-eng" #bucket_name

activities_data = pd.read_csv(s3.get_object(
            Bucket=bucket_name,
            Key="silver/activities/activities_data.csv")['Body'])


activities_data.head(20)


,id,city,name,feature,dist,rate,kinds,lat,lon
0,W51811743,guangzhou,People's Park,Feature,99.415259,3,"urban_environment,gardens_and_parks,cultural,i...",23.130188,113.259049
1,N4258606482,guangzhou,广州原点,Feature,163.945015,2,"historic,monuments_and_memorials,interesting_p...",23.128857,113.258987
2,N5029899501,guangzhou,Department of Finance of Guangdong Province,Feature,358.474138,3,"other,unclassified_objects,interesting_places,...",23.130436,113.263466
3,N2920530014,guangzhou,千年古楼遗址,Feature,428.668104,2,"architecture,historic_architecture,interesting...",23.127922,113.263535
4,W288494693,guangzhou,Archaeological Site Museum of Nanyue Palace,Feature,454.877203,3,"historic,archaeology,cultural,museums,interest...",23.129606,113.264420
5,W146674383,guangzhou,Six Banyan tree Flower pagoda,Feature,549.620645,7,"towers,religion,architecture,buddhist_temples,...",23.130945,113.254730
6,W146674393,guangzhou,Temple of the Six Banyan Trees,Feature,562.639736,7,"religion,buddhist_temples,interesting_places",23.130446,113.254524
7,W288492198,guangzhou,Beijing Road Pedestrian Street,Feature,587.822882,2,"cultural,urban_environment,interesting_places,...",23.125854,113.263588
8,W288492189,guangzhou,City God Temple,Feature,596.159093,3,"religion,other_temples,interesting_places",23.129494,113.265793
9,N4221460719,guangzhou,Archaeological Site of the Wooden Watergate of...,Feature,607.873537,7,"historic,archaeology,interesting_places,other_...",23.124840,113.262024


In [2]:
print("SHAPE (rows, cols):", activities_data.shape)
print("------")

print("INFO:")
print(activities_data.info())
print("------")

print("DESCRIBE:")
print(activities_data.describe())
print("------")

print("NULL COUNTS:")
print(activities_data.isnull().sum())
print("------")

print("DUPLICATE CITY NAMES:", activities_data['city'].duplicated().sum())
print("------")

print("UNIQUE CITIES:", activities_data['city'].nunique())
print(activities_data['city'].unique())
print("------")

SHAPE (rows, cols): (229, 9)
------
INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229 entries, 0 to 228
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       229 non-null    object 
 1   city     229 non-null    object 
 2   name     229 non-null    object 
 3   feature  229 non-null    object 
 4   dist     229 non-null    float64
 5   rate     229 non-null    int64  
 6   kinds    229 non-null    object 
 7   lat      229 non-null    float64
 8   lon      229 non-null    float64
dtypes: float64(3), int64(1), object(5)
memory usage: 16.2+ KB
None
------
DESCRIBE:
              dist        rate         lat         lon
count   229.000000  229.000000  229.000000  229.000000
mean   1884.918842    3.768559   20.728442  114.316204
std    1961.094719    1.864645   15.284892   19.604255
min      33.303803    2.000000   -6.204541   72.816834
25%     718.836575    3.000000   19.042772  106.831085
50%    1211.522805   

There are duplicates in name / near-duplicate names for the same place (e.g. lines 179 and 182,"Edo Castle" vs "Edo Castle Honmaru").

Cause: OpenTripMap pulls from various sources (src_geom) Tried src_geom=osm  still has dupes (confirms it's an OSM thing, not cross-source), wikidata looks cleaner but has less coverage for smaller cities.

Possible fix: round lat/lon (~3 decimals) and keep highest-rated entry per cluster, since exact name matching won't catch stuff like "Edo Castle" vs "Edo Castle Honmaru".

In [6]:
activities_data.sort_values(['city','rate'], ascending= [False,False]).head(30)

,id,city,name,feature,dist,rate,kinds,lat,lon
184,N5562218658,tokyo,Edo Castle Honmaru Goten Palace,Feature,435.672851,7,"fortifications,historic,interesting_places,oth...",35.688004,139.754501
186,N5409598583,tokyo,Edo Castle,Feature,462.804812,7,"fortifications,historic,interesting_places,cas...",35.688374,139.754410
208,W145400030,tokyo,Sakurada gate,Feature,754.718775,7,"fortifications,defensive_walls,historic,intere...",35.678524,139.753952
219,W43935790,tokyo,Shimizu-mon Gate,Feature,864.292559,7,"fortifications,defensive_walls,historic,intere...",35.692734,139.752564
221,N3949450546,tokyo,Site of MOJ Main Building,Feature,882.172670,7,"architecture,historic_architecture,interesting...",35.677235,139.753494
222,N5350845149,tokyo,Instrument Shelter for Origin of the Japan Ver...,Feature,922.804552,7,"architecture,historic_architecture,interesting...",35.677219,139.747803
224,N5834330376,tokyo,Shimizu Gate,Feature,953.787805,7,"fortifications,historic,interesting_places,oth...",35.693432,139.753448
227,W176046390,tokyo,Tayasu Gate,Feature,1022.705268,7,"fortifications,defensive_walls,historic,intere...",35.694057,139.749298
179,W534754971,tokyo,Imperial Palace,Feature,148.166057,3,"palaces,architecture,historic_architecture,his...",35.683811,139.750656
180,R7676266,tokyo,Tokyo Imperial Palace,Feature,148.166057,3,"palaces,architecture,historic_architecture,int...",35.683811,139.750656
